**Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# UK Carbon Intensity API Example

This notebook demonstrates how to fetch and visualize real-time carbon intensity data from the UK's National Grid ESO API.

**Carbon Intensity** measures how much CO₂ is produced per unit of electricity consumed. Lower values indicate cleaner energy (more renewables), while higher values indicate more fossil fuel use.

## Getting Started

First, we'll import the libraries we need:

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd
import requests
import altair as alt

## Building the API URL

The Carbon Intensity API follows a predictable URL structure that we can build programmatically:

**Base URL structure:**
```
https://api.carbonintensity.org.uk/intensity/{date}/{period}
```

**Components:**
- `{date}`: Start date in YYYY-MM-DD format
- `{period}`: Time period - we'll use `fw48h` (forward 48 hours)

Let's construct our URL step by step:

In [ ]:
# Step 1: Get today's date in the required format (YYYY-MM-DD)
today = pd.Timestamp.now().strftime('%Y-%m-%d')
print(f"Today's date: {today}")

# Step 2: Define the API components
base_url = "https://api.carbonintensity.org.uk/intensity"
period = "fw48h"  # Forward 48 hours from the start date

# Step 3: Build the complete URL
url = f"{base_url}/{today}/{period}"
print(f"Complete API URL: {url}")

# Let's break down what this URL means:
print(f"\nURL breakdown:")
print(f"  Base: {base_url}")
print(f"  Start date: {today}")
print(f"  Period: {period} (next 48 hours)")
print(f"  → This will get carbon intensity forecasts from {today} for the next 2 days")

## Making the API Request

With our URL constructed, let's make the API call and retrieve the data:

In [ ]:
# Now fetch the data using our constructed URL
response = requests.get(url)

# Convert the JSON response to a pandas DataFrame
carbon_data = pd.json_normalize(response.json()['data'])

# Show the first few rows
carbon_data.head()

## Data Preparation for Visualization

Let's clean up the data and prepare it for charting:

In [ ]:
# Clean and prepare the data
carbon_clean = carbon_data.copy()

# Convert timestamps to datetime
carbon_clean['from_dt'] = pd.to_datetime(carbon_clean['from'])
carbon_clean['to_dt'] = pd.to_datetime(carbon_clean['to'])

# Extract the intensity values and index (forecast vs actual)
carbon_clean['intensity_forecast'] = carbon_clean['intensity.forecast']
carbon_clean['intensity_index'] = carbon_clean['intensity.index']

# Show the cleaned data structure
print("Data columns:", carbon_clean.columns.tolist())
print("\nFirst few rows:")
carbon_clean[['from_dt', 'intensity_forecast', 'intensity_index']].head()

## Visualizing Carbon Intensity Over Time

Now let's create a simple line chart showing how carbon intensity changes over the next 48 hours:

In [ ]:
# Create a line chart of carbon intensity over time
chart = alt.Chart(carbon_clean).mark_line(
    point=True,
).encode(
    x=alt.X('from_dt:T').title('Time').axis(format='%H:%M'),
    y=alt.Y('intensity_forecast:Q').title('Carbon Intensity (gCO₂/kWh)').scale(zero=False),
    tooltip=[
        alt.Tooltip('from_dt:T').title('Time').format('%d %b %H:%M'),
        alt.Tooltip('intensity_forecast:Q').title('Carbon Intensity'),
        alt.Tooltip('intensity_index:O').title('Intensity Level')
    ]
).properties(
    title='UK Carbon Intensity Forecast (Next 48 Hours)',
    width=600,
    height=300
)

chart